In [6]:
# Read 1000A-2027_de.csv
import re, io, csv, pathlib
import pandas as pd
import numpy as np

_12DIG = re.compile(r"^\d{12}$")

# ---------- robust text read ----------
def _read_text_utf8(path, prefer_sep=";"):
    b = pathlib.Path(path).read_bytes()
    # utf-8-sig eats the BOM if present; fallback to utf-8
    try:
        text = b.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = b.decode("utf-8", errors="ignore")

    # normalize line endings and strip NUL/NBSP
    if "\n" not in text and "\r" in text:
        text = text.replace("\r", "\n")
    text = text.replace("\r\n", "\n")
    text = text.replace("\x00", "").replace("\u00a0", " ")

    # verify sep (you said it's ';' but let's be safe on weird files)
    head = "\n".join(text.splitlines()[:10])
    if head.count(prefer_sep) < 3:
        counts = {s: head.count(s) for s in [";", "\t", ","]}
        prefer_sep = max(counts, key=counts.get)

    return text, prefer_sep

def _read_csv_anycols(text: str, sep: str) -> pd.DataFrame:
    # compute max field count to force stable column width
    lines = text.splitlines()
    # ignore very short metadata lines when computing width, but keep them in the CSV
    max_fields = max((ln.count(sep) for ln in lines[0:2000]), default=0) + 1
    names = list(range(max_fields))
    return pd.read_csv(
        io.StringIO(text),
        sep=sep,
        header=None,
        names=names,            # fixes "Expected 1 fields..." by padding short rows
        dtype=str,
        quoting=csv.QUOTE_NONE, # no special quoting in these exports
        keep_default_na=False,  # keep "-" etc., we'll handle ourselves
        na_filter=False,
        low_memory=False,
    )

# ---------- structure helpers ----------
def _row_has_many_geo_codes(row: pd.Series) -> bool:
    c = 0
    for cell in row.fillna(""):
        parts = str(cell).split()
        if parts and _12DIG.match(parts[0]):
            c += 1
    return c >= 3

def _find_block_bounds(df: pd.DataFrame):
    first_data = df.index[(df[0] == "Insgesamt") & (df[1] == "Insgesamt")]
    if len(first_data) == 0:
        raise ValueError("Couldn't find 'Insgesamt / Insgesamt' start.")
    start_total = int(first_data[0])

    male_hdr = df.index[(df[0] == "Männlich") & (df[1] == "Insgesamt")]
    female_hdr = df.index[(df[0] == "Weiblich") & (df[1] == "Insgesamt")]
    if len(male_hdr) == 0 or len(female_hdr) == 0:
        raise ValueError("Couldn't find Männlich/Weiblich markers.")
    start_male, start_female = int(male_hdr[0]), int(female_hdr[0])

    footer_idx = df.index[df[0].fillna("").astype(str).str.startswith(("©","Stand:","__________"))]
    footer = int(footer_idx[0]) if len(footer_idx) else len(df)

    return {
        "total":  (start_total,   start_male - 1),
        "male":   (start_male + 1, start_female - 1),
        "female": (start_female + 1, footer - 1),
    }, start_total

def _build_geo_schema(df: pd.DataFrame, data_start_row: int):
    """
    Parse the geo header row (with ARS + name) and the sub-header row (with 'Anzahl', '%', 'e').
    Return schema entries ONLY for value columns (Anzahl or %, but we'll filter to Anzahl later),
    and separately remember the position of the flag column so we can ignore it.
    We DO NOT drop any geos based on flags.
    """
    # find the GEO header line above the first data row
    geo_line_idx = None
    for r in range(data_start_row - 1, max(0, data_start_row - 10), -1):
        if _row_has_many_geo_codes(df.loc[r]):
            geo_line_idx = r
            break
    if geo_line_idx is None:
        raise ValueError("Geo header line not found.")

    sub_line_idx = geo_line_idx + 1
    geo_line = df.loc[geo_line_idx].fillna("")
    sub_line = df.loc[sub_line_idx].fillna("")

    schema = []
    c = 2  # data columns start at 2 (0/1 are group + age)
    max_c = df.shape[1]

    while c < max_c:
        geo_cell = str(geo_line.iloc[c]).strip()
        sub = str(sub_line.iloc[c]).strip().lower()

        if not geo_cell:
            c += 1
            continue

        # Determine geo_id (Deutschland has no 12-digit code)
        parts = geo_cell.split()
        geo_id = parts[0] if (parts and _12DIG.match(parts[0])) else "DE"

        # Only consider actual value columns here; ignore pure flag columns.
        if sub.startswith("anzahl") or sub.startswith("%"):
            measure = "share" if sub.startswith("%") else "anzahl"

            # If the next column in the sub-line is 'e', treat it as the flag column and skip it
            flag_idx = c + 1 if (c + 1 < max_c and str(sub_line.iloc[c + 1]).strip().lower() in {"e", ""}) else None

            schema.append({
                "value_col": c,
                "flag_col": flag_idx,   # may be None
                "geo_id": geo_id,
                "measure": measure,
            })

            # advance: skip the neighboring flag column if present
            c = (flag_idx + 1) if flag_idx is not None else (c + 1)
        else:
            # This column isn’t a value (likely the 'e' flag); skip it
            c += 1

    return schema

def _clean_num_zero(s):
    if s is None:
        return 0.0
    s = str(s).strip()
    if s in {"", "-", "–"}:
        return 0.0
    s = s.replace(",", ".")
    try:
        return float(s)
    except Exception:
        return 0.0

def _block_to_matrix(df: pd.DataFrame, start: int, end: int, schema):
    """
    Build a wide matrix for one block (Total/Male/Female):
    - rows: ages
    - cols: geo_ids
    - values: Anzahl (value columns only)
    Flag columns are ignored; we do NOT drop geos because of flags.
    """
    # collect unique ages (in file order)
    ages = []
    age_rows = []
    for r in range(start, end + 1):
        age = str(df.iat[r, 1]).strip()
        if not age or age.lower().startswith("davon"):
            continue
        ages.append(age)
        age_rows.append(r)

    # initialize data arrays for each geo_id found in schema (Anzahl only)
    value_specs = [s for s in schema if s["measure"] == "anzahl"]
    geo_ids = [s["geo_id"] for s in value_specs]
    geo_ids_unique = list(dict.fromkeys(geo_ids))  # preserve order, remove dups

    data = {gid: [0.0] * len(ages) for gid in geo_ids_unique}

    # fill values
    for idx, r in enumerate(age_rows):
        for s in value_specs:
            gid = s["geo_id"]
            c = s["value_col"]
            # robust numeric parse: '-' → 0, comma decimals → dot
            val_raw = df.iat[r, c]
            v = _clean_num_zero(val_raw)
            data[gid][idx] = v

    mat = pd.DataFrame(data, index=ages)
    mat.index.name = "age_label"
    return mat.sort_index(axis=1)

# ---------- public API ----------
def load_age_csv_to_matrices(path: str):
    text, sep = _read_text_utf8(path, prefer_sep=";")
    raw = _read_csv_anycols(text, sep)
    # drop empty rows/cols and trim whitespace
    raw = raw.dropna(how="all", axis=0).dropna(how="all", axis=1).reset_index(drop=True)
    raw = raw.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    bounds, first_data_row = _find_block_bounds(raw)
    schema = _build_geo_schema(raw, first_data_row)

    total_df  = _block_to_matrix(raw, *bounds["total"],  schema)
    male_df   = _block_to_matrix(raw, *bounds["male"],   schema)
    female_df = _block_to_matrix(raw, *bounds["female"], schema)

    # optional: ensure identical row order across all three
    all_ages = total_df.index.union(male_df.index).union(female_df.index)
    total_df  = total_df.reindex(all_ages)
    male_df   = male_df.reindex(all_ages)
    female_df = female_df.reindex(all_ages)

    return total_df, male_df, female_df


total_df, male_df, female_df = load_age_csv_to_matrices(
    r"C:\Users\petre\PycharmProjects\cleancensus\additional_data\gender\1000A-2027_de\1000A-2027_de.csv"
)
total_df  = total_df.drop(columns=["DE"], errors="ignore")
male_df   = male_df.drop(columns=["DE"], errors="ignore")
female_df = female_df.drop(columns=["DE"], errors="ignore")

total_df.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_totals.pickle")
male_df.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_males.pickle")
female_df.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_females.pickle")

print(total_df.shape, male_df.shape, female_df.shape)

# geos present in all three?
common = set(total_df.columns) & set(male_df.columns) & set(female_df.columns)
only_in_total  = set(total_df.columns)  - common
only_in_male   = set(male_df.columns)   - common
only_in_female = set(female_df.columns) - common
print("only_in_total:",  len(only_in_total))
print("only_in_male:",   len(only_in_male))
print("only_in_female:", len(only_in_female))

# if you want strict alignment for the residual check:
all_geos = sorted(set(total_df.columns) | set(male_df.columns) | set(female_df.columns))
all_ages = sorted(set(total_df.index)   | set(male_df.index)   | set(female_df.index))

total  = total_df .reindex(index=all_ages, columns=all_geos, fill_value=0.0)
male   = male_df  .reindex(index=all_ages, columns=all_geos, fill_value=0.0)
female = female_df.reindex(index=all_ages, columns=all_geos, fill_value=0.0)

diff = total - (male + female)
print("max abs residual:", diff.abs().to_numpy().max())
print("example 010010000000 present in female?:", "010010000000" in female.columns)


C:\Users\petre\AppData\Local\Temp\ipykernel_36444\1131383290.py:194: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  raw = raw.applymap(lambda x: x.strip() if isinstance(x, str) else x)


(102, 10786) (102, 10786) (102, 10786)
only_in_total: 0
only_in_male: 0
only_in_female: 0
max abs residual: nan
example 010010000000 present in female?: True


In [7]:
import re
import pandas as pd
import numpy as np


total = pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_totals.pickle")
male=pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_males.pickle")
female=pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_females.pickle")

# --- 2) Rename age rows to numeric (keep 'Insgesamt')
_age_re = re.compile(r"^\s*(\d+)\s+Jahr")

def _canon_age(lbl: str):
    s = str(lbl).strip()
    sl = s.lower()
    if sl.startswith("unter 1 jahr"):          return "0"
    if sl.startswith("100 jahre und älter"):   return "100"
    if sl == "insgesamt":                      return "Insgesamt"
    m = _age_re.match(s)
    if m:                                      return str(int(m.group(1)))
    raise ValueError(f"Unrecognized age label: {lbl!r}")

def _rename(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = [ _canon_age(x) for x in out.index ]
    # order ages 0..100 then 'Insgesamt' (if present)
    ages = [str(i) for i in range(0, 101)]
    order = [a for a in ages if a in out.index]
    if "Insgesamt" in out.index: order.append("Insgesamt")
    return out.reindex(order)

total  = _rename(total)
male   = _rename(male)
female = _rename(female)

total.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_totals.pickle")
male.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_males.pickle")
female.to_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_females.pickle")

# --- 3) Verify identical labels (you said shapes already match; we assert)
assert total.shape == male.shape == female.shape, "Shapes differ."
assert total.index.equals(male.index) and total.index.equals(female.index), "Row labels differ."
assert total.columns.equals(male.columns) and total.columns.equals(female.columns), "Column labels differ."

# --- 4) Residuals
combined = male + female
diff = total - combined

tol = 1e-6
abs_diff = diff.abs()
n_bad = int((abs_diff > tol).to_numpy().sum())
max_val = abs_diff.to_numpy().max() if n_bad else 0.0
print(f"[CHECK] |total - (male+female)| > {tol}: {n_bad} cells; max abs diff = {max_val:.6f}")

if n_bad:
    # worst single cell
    idxmax = np.unravel_index(abs_diff.values.argmax(), diff.shape)
    worst_age = diff.index[idxmax[0]]
    worst_geo = diff.columns[idxmax[1]]
    print(f"   worst cell: age='{worst_age}', geo='{worst_geo}', diff={diff.iat[idxmax[0], idxmax[1]]:.6f}")

    # per-geo and per-age aggregates (helpful to spot systematic issues)
    per_geo = diff.sum(axis=0).abs().sort_values(ascending=False).head(10)
    per_age = diff.sum(axis=1).abs().sort_values(ascending=False).head(10)
    print("\nTop 10 |sum over ages| per geo:\n", per_geo.to_string())
    print("\nTop 10 |sum over geos| per age:\n", per_age.to_string())
else:
    print("All good: totals equal male+female within tolerance.")

# --- 5) Save full residual matrix
out_csv = r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_total_minus_male_female.csv"
diff.to_csv(out_csv)
print(f"\nSaved full diff matrix to: {out_csv}")


[CHECK] |total - (male+female)| > 1e-06: 826693 cells; max abs diff = nan
   worst cell: age='Insgesamt', geo='010010000000', diff=nan

Top 10 |sum over ages| per geo:
 145220020020    100.0
073405001021     89.0
091870117117     88.0
072325005042     83.0
120675701528     83.0
160630092092     82.0
072315001077     82.0
066330011011     80.0
051620024024     79.0
096770155155     78.0

Top 10 |sum over geos| per age:
 13    585.0
89    559.0
48    533.0
62    512.0
47    452.0
15    430.0
2     420.0
72    411.0
17    407.0
5     403.0

Saved full diff matrix to: C:\Users\petre\PycharmProjects\cleancensus\with_gender\gemeinde_total_minus_male_female.csv


In [9]:


import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender")
DIFF_CSV = BASE / "gemeinde_total_minus_male_female.csv"

# 1) Load residual matrix: rows=ages ('0'..'100','Insgesamt'), cols=geo IDs
diff = pd.read_csv(DIFF_CSV, index_col=0)
# ensure numeric
diff = diff.apply(pd.to_numeric, errors="coerce").fillna(0.0)


# 2) Overall stats
tol = 1e-6
abs_diff = diff.abs()
n_rows, n_cols = diff.shape
n_cells = n_rows * n_cols
n_bad = int((abs_diff > tol).to_numpy().sum())
share_bad = n_bad / n_cells

max_val = abs_diff.to_numpy().max()
min_val = diff.to_numpy().min()
max_pos = diff.to_numpy().max()

# where is worst absolute deviation?
ij = np.unravel_index(abs_diff.values.argmax(), diff.shape)
worst_age = diff.index[ij[0]]
worst_geo = diff.columns[ij[1]]
worst_val = diff.iat[ij[0], ij[1]]

print("=== OVERALL ===")
print(f"shape: {n_rows} ages x {n_cols} geos ({n_cells:,} cells)")
print(f"|diff| > {tol}: {n_bad:,} cells ({share_bad:.2%})")
print(f"max |diff|: {max_val:.3f}  (worst at age='{worst_age}', geo='{worst_geo}', diff={worst_val:.3f})")
print(f"min diff: {min_val:.3f}   max diff: {max_pos:.3f}")

# 3) Per-geo summaries
per_geo_sum      = diff.sum(axis=0)          # signed bias per geo
per_geo_abs_sum  = abs_diff.sum(axis=0)      # magnitude over all ages
per_geo_max_abs  = abs_diff.max(axis=0)      # worst age for each geo
per_geo_nonzero  = (abs_diff > tol).sum(axis=0)

print("\n=== TOP GEO OFFENDERS (by |sum over ages|) ===")
print(per_geo_abs_sum.sort_values(ascending=False).head(15).to_string())

print("\n=== TOP GEO WORST SINGLE-AGE DEVIATION (by max |diff|) ===")
print(per_geo_max_abs.sort_values(ascending=False).head(15).to_string())

# 4) Per-age summaries
per_age_sum      = diff.sum(axis=1)          # signed bias per age across geos
per_age_abs_sum  = abs_diff.sum(axis=1)
per_age_max_abs  = abs_diff.max(axis=1)
per_age_nonzero  = (abs_diff > tol).sum(axis=1)

print("\n=== TOP AGE OFFENDERS (by |sum over geos|) ===")
print(per_age_abs_sum.sort_values(ascending=False).head(15).to_string())

print("\n=== TOP AGES WORST SINGLE-GEO (by max |diff|) ===")
print(per_age_max_abs.sort_values(ascending=False).head(15).to_string())

# 5) Specific checks that often reveal parsing issues
if "Insgesamt" in diff.index:
    inz = diff.loc["Insgesamt"]
    print("\n=== 'Insgesamt' row check ===")
    print(f"  max |diff|: {inz.abs().max():.3f},  #nonzero geos: {(inz.abs() > tol).sum()}/{inz.size}")

# columns/ages that are entirely zero (good = perfect match, really not needed, just for reference)
zero_geos = (abs_diff > tol).sum(axis=0) == 0
zero_ages = (abs_diff > tol).sum(axis=1) == 0
print(f"\nGeos with perfect match (all ages): {int(zero_geos.sum())}/{n_cols}")
print(f"Ages with perfect match (all geos): {int(zero_ages.sum())}/{n_rows}")

# 6) Write helper CSVs for deeper inspection
(per_geo_abs_sum.rename("abs_sum_over_ages")
          .to_csv(BASE / "residuals_per_geo_abs_sum.csv"))
(per_geo_max_abs.rename("max_abs_over_ages")
          .to_csv(BASE / "residuals_per_geo_max_abs.csv"))
(per_age_abs_sum.rename("abs_sum_over_geos")
          .to_csv(BASE / "residuals_per_age_abs_sum.csv"))
(per_age_max_abs.rename("max_abs_over_geos")
          .to_csv(BASE / "residuals_per_age_max_abs.csv"))

print(f"\nSaved details to:\n- {BASE/'residuals_per_geo_abs_sum.csv'}"
      f"\n- {BASE/'residuals_per_geo_max_abs.csv'}"
      f"\n- {BASE/'residuals_per_age_abs_sum.csv'}"
      f"\n- {BASE/'residuals_per_age_max_abs.csv'}")


C:\Users\petre\AppData\Local\Temp\ipykernel_36444\1804157389.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  diff = pd.read_csv(DIFF_CSV, index_col=0)


=== OVERALL ===
shape: 102 ages x 10786 geos (1,100,172 cells)
|diff| > 1e-06: 826,693 cells (75.14%)
max |diff|: 8.000  (worst at age='2', geo='130765665154', diff=-8.000)
min diff: -8.000   max diff: 8.000

=== TOP GEO OFFENDERS (by |sum over ages|) ===
010550042042    249.0
010550004004    248.0
010530032032    248.0
084355007059    247.0
053700020020    247.0
094740126126    247.0
091800117117    247.0
084265005070    246.0
081175006049    245.0
010020000000    245.0
057620020020    245.0
034510005005    245.0
034590005005    244.0
095720135135    244.0
097715705146    243.0

=== TOP GEO WORST SINGLE-AGE DEVIATION (by max |diff|) ===
010615138035    8.0
073345005006    8.0
130765665154    8.0
071415009023    8.0
010535318048    7.0
100450116116    7.0
010585893052    7.0
091865157130    7.0
010595993092    7.0
094775435151    7.0
083355002026    7.0
032565410017    7.0
095770114114    7.0
096790201201    7.0
091780143143    7.0

=== TOP AGE OFFENDERS (by |sum over geos|) ===
58    

Damit passen Ages und Gender (Gemeinderohdaten) - diffs zwischen total und female+male stammen direkt aus den Zensusdaten und sind extrem vernachlässigbar (einstellig auch in Großstadt).
Damit folgt matching zu den 100m Zellen und application der gender ratios.

In [2]:
# Merge Gemeinde (and Kreis etc) info onto census 100m cells  → Parquet (streaming) instead of pickle as previously, mostly for size

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

# ----------------------------
# Config
# ----------------------------
CELLS_IN = Path(r"C:\Users\petre\PycharmProjects\cleancensus\merged\df100_with_single_years.pickle")
CELLS_ID_COL = "GITTER_ID_100m"

GEM_GPKG = Path(r"C:\Users\petre\PycharmProjects\cleancensus\additional_data\gender\vg250_12-31.utm32s.gpkg.ebenen\vg250_ebenen_1231\DE_VG250.gpkg")
GEM_LAYER = "v_vg250_gem"

OUT_PARQUET = Path(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gemeinde.parquet")
CHUNK_SIZE = 1_000_000

EPSG_POINTS = 3035   # cell IDs are LAEA Europe (EPSG:3035)
EPSG_GEM    = 25832  # UTM32N (expected for GeoPackage)

# ----------------------------
# Helpers
# ----------------------------

def normalize_objects(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object":
            s = df[col]

            # 1) decode bytes → str
            if s.map(lambda x: isinstance(x, (bytes, bytearray))).any():
                s = s.map(lambda x: x.decode("utf-8", "replace") if isinstance(x, (bytes, bytearray)) else x)

            # 2) try numeric if it looks numeric for most rows
            num = pd.to_numeric(s, errors="coerce")
            if num.notna().sum() >= 0.95 * s.notna().sum():
                df[col] = num
            else:
                # fall back to string (Arrow-friendly)
                df[col] = s.astype("string[pyarrow]")
    return df

def parse_centroids_3035(df: pd.DataFrame, id_col: str) -> gpd.GeoDataFrame:
    """From IDs like 'CRS3035RES100mN2689100E4337000' -> centroid points (+50,+50)."""
    ne = df[id_col].astype(str).str.extract(r"N(\d+)E(\d+)", expand=True)
    if ne.isna().any().any():
        bad = df.loc[ne.isna().any(axis=1), id_col].head(5).tolist()
        raise ValueError(f"Could not parse N/E from: {bad}")
    n = ne[0].astype(np.int64) + 50
    e = ne[1].astype(np.int64) + 50
    return gpd.GeoDataFrame(df[[id_col]].copy(),
                            geometry=gpd.points_from_xy(e, n, crs=f"EPSG:{EPSG_POINTS}"))

def require_exact_12_digit_key(gem: gpd.GeoDataFrame) -> str:
    """Validate Gemeinde key column: exactly 12 numeric digits. No padding/transform."""
    col = "Regionalschlüssel_ARS"  # as in your dataset
    if col not in gem.columns:
        raise KeyError(f"Missing '{col}' on Gemeinde layer.")
    s = gem[col].astype(str)
    ok = s.str.fullmatch(r"\d{12}")
    if not ok.all():
        bad = gem.loc[~ok, [col]].head(5)
        raise ValueError(f"Gemeinde key '{col}' must be exactly 12 digits. Examples:\n{bad}")
    return col

def add_ars_parts(gem: gpd.GeoDataFrame, key_col: str) -> gpd.GeoDataFrame:
    """Derive fixed-length parts from the already-validated 12-digit key."""
    ars = gem[key_col].astype(str)
    gem = gem.copy()
    gem["RegionalSchlüssel_ARS"]        = ars
    gem["Land"]                         = ars.str.slice(0, 2)
    gem["Regierungsbezirk"]             = ars.str.slice(2, 3)
    gem["Kreis"]                        = ars.str.slice(3, 5)
    gem["VerwaltungsgemeinschaftTeil1"] = ars.str.slice(5, 7)
    gem["VerwaltungsgemeinschaftTeil2"] = ars.str.slice(7, 9)
    gem["Gemeinde"]                     = ars.str.slice(9, 12)

    expected = {
        "RegionalSchlüssel_ARS": 12, "Land": 2, "Regierungsbezirk": 1, "Kreis": 2,
        "VerwaltungsgemeinschaftTeil1": 2, "VerwaltungsgemeinschaftTeil2": 2, "Gemeinde": 3
    }
    for c, L in expected.items():
        if not (gem[c].astype(str).str.len() == L).all():
            raise ValueError(f"{c} has wrong length; expected {L}.")
    return gem

# ----------------------------
# Load Gemeinden (EPSG:25832 -> 3035)
# ----------------------------
gem = gpd.read_file(GEM_GPKG, layer=GEM_LAYER)
if gem.crs is None:
    raise ValueError("Gemeinde layer has no CRS. Expected EPSG:25832.")
if gem.crs.to_epsg() != EPSG_GEM:
    raise ValueError(f"Gemeinde layer CRS is {gem.crs}, expected EPSG:{EPSG_GEM} (UTM32N).")

key_col = require_exact_12_digit_key(gem)
gem = gem.to_crs(EPSG_POINTS)
gem = add_ars_parts(gem, key_col)
gem = gem[[
    "RegionalSchlüssel_ARS", "Land", "Regierungsbezirk", "Kreis",
    "VerwaltungsgemeinschaftTeil1", "VerwaltungsgemeinschaftTeil2",
    "Gemeinde", "geometry"
]].copy()
_ = gem.sindex  # build once

# ----------------------------
# Load full cells and stream join → Parquet (append via writer)
# ----------------------------
cells_all = pd.read_pickle(CELLS_IN)
if CELLS_ID_COL not in cells_all.columns:
    raise KeyError(f"Missing column '{CELLS_ID_COL}' in cells pickle.")

# Define output column order: full original cols + new attrs
new_cols = ["RegionalSchlüssel_ARS","Land","Regierungsbezirk","Kreis",
            "VerwaltungsgemeinschaftTeil1","VerwaltungsgemeinschaftTeil2","Gemeinde"]
out_cols = list(cells_all.columns) + new_cols

# define which columns must be STRING (Arrow large_string)
FORCE_STRING_COLS = {
    "GITTER_ID_100m", "GITTER_ID_1km", "GITTER_ID_10km",
    "RegionalSchlüssel_ARS", "Land", "Regierungsbezirk", "Kreis",
    "VerwaltungsgemeinschaftTeil1", "VerwaltungsgemeinschaftTeil2", "Gemeinde",
}
# add all werterläuternde columns dynamically
FORCE_STRING_COLS |= {c for c in cells_all.columns if c.startswith("werterlaeuternde_Zeichen_")}

def coerce_for_arrow(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # force specific cols to Arrow strings (large_string)
    for c in FORCE_STRING_COLS:
        if c in df.columns:
            df[c] = df[c].astype("string[pyarrow]")
    # keep booleans stable
    if "is_orphan" in df.columns:
        df["is_orphan"] = df["is_orphan"].astype("bool")
    return df


writer = None
writer = None
schema = None

for start in tqdm(range(0, len(cells_all), CHUNK_SIZE)):
    chunk_full = cells_all.iloc[start:start+CHUNK_SIZE, :].copy()
    pts = parse_centroids_3035(chunk_full, CELLS_ID_COL)
    joined = gpd.sjoin(pts, gem, how="left", predicate="within")
    joined = joined.drop(columns=["index_left", "index_right"], errors="ignore")
    attrs = pd.DataFrame(joined.drop(columns="geometry"))[[CELLS_ID_COL,
        "RegionalSchlüssel_ARS","Land","Regierungsbezirk","Kreis",
        "VerwaltungsgemeinschaftTeil1","VerwaltungsgemeinschaftTeil2","Gemeinde"]]
    enriched = chunk_full.merge(attrs, on=CELLS_ID_COL, how="left")

    # enforce stable dtypes
    enriched = coerce_for_arrow(enriched)
    # (optional) any remaining object cols that should be numbers:
    # enriched[num_cols] = enriched[num_cols].apply(pd.to_numeric, errors="coerce")

    table = pa.Table.from_pandas(enriched[out_cols], preserve_index=False, schema=schema)
    if writer is None:
        # freeze schema from the *first* chunk
        schema = table.schema
        writer = pq.ParquetWriter(OUT_PARQUET.as_posix(), schema, compression="zstd")
    writer.write_table(table)

writer.close()


print(f"Done. Wrote {OUT_PARQUET}")


100%|██████████| 4/4 [01:00<00:00, 15.01s/it]

Done. Wrote 0 rows → C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gemeinde.parquet


In [3]:
df = pd.read_parquet(r"C:\Users\petre\PycharmProjects\cleancensus\with_gender\cells_100m_with_gemeinde.parquet")
print(">>> df.shape:", df.shape)
print(">>> df.columns:", df.columns)
print(">>> df.head():")
print(df.head())
# 3,148,482 rows

>>> df.shape: (3148482, 337)
>>> df.columns: Index(['GITTER_ID_100m',
       'Insgesamt_Bevoelkerung_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'Unter10_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a10bis19_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a20bis29_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a30bis39_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a40bis49_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a50bis59_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a60bis69_Alter_in_10er-Jahresgruppen_100m-Gitter',
       'a70bis79_Alter_in_10er-Jahresgruppen_100m-Gitter',
       ...
       'AGE_98', 'AGE_99', 'AGE_100', 'RegionalSchlüssel_ARS', 'Land',
       'Regierungsbezirk', 'Kreis', 'VerwaltungsgemeinschaftTeil1',
       'VerwaltungsgemeinschaftTeil2', 'Gemeinde'],
      dtype='object', length=337)
>>> df.head():
                   GITTER_ID_100m  \
0  CRS3035RES100mN2689100E4337000   
1  CRS3035RES100mN2689100E4341100   
2  CRS3035RES100mN269080